# KAN-Based Deepfake Detection via Phase-Spectrum Analysis

| | |
|---|---|
| **Dataset** | DeepDetect-2025 (~115k images — Midjourney, SD3, DALL·E 3) |
| **Model** | Kolmogorov–Arnold Network (KAN) on flattened phase spectrum |
| **Target** | Binary classification — Real (0) vs Fake (1) |
| **Venue** | IEEE journal / conference submission |
| **Runtime** | Google Colab (≤ 12 GB RAM, T4 / L4 GPU) |

### Key Design Decisions
- **NO** `torchvision.transforms.Resize` — resizing destroys high-frequency artifacts
- Images are centre-cropped (224×224), converted to grayscale, then 2-D FFT phase spectrum is extracted
- KAN replaces standard linear layers with learnable B-spline edge functions

## Cell 1 — Environment Setup, Installs & Deterministic Seeds

In [ ]:
import subprocess, sys

def _pip(pkg: str) -> None:
    """Install a package silently if not already present."""
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )

# Ensure kagglehub is available (Colab may not have it by default)
try:
    import kagglehub
except ImportError:
    _pip("kagglehub")
    import kagglehub

# scikit-learn for metrics
try:
    import sklearn
except ImportError:
    _pip("scikit-learn")

import os, random, time, json, warnings
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms as T
from PIL import Image
import matplotlib
matplotlib.use("Agg")             # headless backend for Colab
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve, confusion_matrix,
    ConfusionMatrixDisplay, classification_report,
)
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)

# ── deterministic seeds ──────────────────────────────────────────────────────
SEED = 42

def seed_everything(seed: int = SEED) -> None:
    """Set all random seeds for full reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Python  : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {DEVICE}")
print(f"Seed    : {SEED}")
print("Environment ready ✅")

## Cell 2 — Download Dataset & Build File Manifest

In [ ]:
dataset_path = kagglehub.dataset_download("ayushmandatta1/deepdetect-2025")
DATASET_ROOT = Path(dataset_path)
print(f"Dataset root: {DATASET_ROOT}")

# ── discover all images and assign labels ────────────────────────────────────
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif", ".webp"}


def discover_images(root: Path) -> List[Tuple[str, int]]:
    """
    Walk the dataset tree and infer binary labels from folder names.
    The DeepDetect-2025 dataset is organised as:
       root / Real / ...
       root / Fake / ...
    Returns a list of (image_path, label) tuples.
    label = 0 → Real, label = 1 → Fake
    """
    samples: List[Tuple[str, int]] = []
    for dirpath, _, filenames in os.walk(root):
        dp_lower = dirpath.lower()
        # Determine label from path components
        if "real" in dp_lower:
            label = 0
        elif "fake" in dp_lower or "ai" in dp_lower or "synthetic" in dp_lower:
            label = 1
        else:
            # Try to infer: if path contains known generator names → Fake
            generators = ["midjourney", "sd3", "stable", "dall", "dalle"]
            if any(g in dp_lower for g in generators):
                label = 1
            else:
                continue  # skip ambiguous directories

        for fn in filenames:
            if Path(fn).suffix.lower() in IMG_EXTS:
                samples.append((os.path.join(dirpath, fn), label))
    return samples


all_samples = discover_images(DATASET_ROOT)
random.shuffle(all_samples)

# ── balance & cap (OOM prevention) ──────────────────────────────────────────
MAX_SAMPLES_PER_CLASS = 25_000              # ≤50 k total — memory safe

reals = [(p, l) for p, l in all_samples if l == 0]
fakes = [(p, l) for p, l in all_samples if l == 1]
n_per_class = min(len(reals), len(fakes), MAX_SAMPLES_PER_CLASS)
reals = reals[:n_per_class]
fakes = fakes[:n_per_class]
all_samples = reals + fakes
random.shuffle(all_samples)

print(f"Total images discovered : {len(reals) + len(fakes)}")
print(f"  Real : {len(reals)}")
print(f"  Fake : {len(fakes)}")
print("Dataset manifest built ✅")

## Cell 3 — Phase-Spectrum Dataset (Lazy-Loading, No Resize)

In [ ]:
CROP_SIZE = 224   # CenterCrop — NO Resize / interpolation


class PhaseSpectrumDataset(Dataset):
    """
    Lazy-loading dataset that:
      1) Reads an image from disk (PIL)
      2) Converts to grayscale
      3) Centre-crops to (CROP_SIZE × CROP_SIZE)
      4) Computes the 2-D FFT, shifts zero-freq to centre, extracts phase
      5) Normalises phase to [0, 1]
      6) Returns (phase_tensor, label)

    IMPORTANT: No Resize transform — resizing destroys the high-frequency
    artifacts that distinguish AI-generated images.
    """

    def __init__(
        self,
        samples: List[Tuple[str, int]],
        crop_size: int = CROP_SIZE,
    ):
        super().__init__()
        self.samples = samples
        self.crop_size = crop_size
        # Deterministic spatial transform — crop only, no resize
        self.spatial_tf = T.Compose([
            T.CenterCrop(crop_size),
        ])

    def __len__(self) -> int:
        return len(self.samples)

    @staticmethod
    def _extract_phase(gray_tensor: torch.Tensor) -> torch.Tensor:
        """
        Compute 2-D FFT → shift → extract phase spectrum.
        Input : (1, H, W) float32 tensor
        Output: (1, H, W) float32 tensor, normalised to [0, 1]
        """
        # Remove channel dim for fft2
        x = gray_tensor.squeeze(0)                      # (H, W)
        fft2 = torch.fft.fft2(x)                        # complex
        fft2_shifted = torch.fft.fftshift(fft2)          # shift DC to centre
        phase = torch.angle(fft2_shifted)                # ∈ [-π, π]
        # Normalise to [0, 1]
        phase = (phase + torch.pi) / (2 * torch.pi)
        return phase.unsqueeze(0)                        # (1, H, W)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert("L")          # grayscale
        except Exception:
            # Fallback: return a black image if file is corrupted
            img = Image.new("L", (self.crop_size, self.crop_size), 0)

        # Ensure image is large enough for CenterCrop
        w, h = img.size
        if w < self.crop_size or h < self.crop_size:
            # Pad to minimum size WITHOUT interpolation/resizing
            new_w = max(w, self.crop_size)
            new_h = max(h, self.crop_size)
            padded = Image.new("L", (new_w, new_h), 0)
            padded.paste(img, ((new_w - w) // 2, (new_h - h) // 2))
            img = padded

        img = self.spatial_tf(img)                       # CenterCrop
        tensor = T.ToTensor()(img)                       # (1, H, W), [0,1]
        phase = self._extract_phase(tensor)              # (1, H, W), [0,1]
        return phase, torch.tensor(label, dtype=torch.float32)


# ── quick sanity check ──────────────────────────────────────────────────────
_ds_test = PhaseSpectrumDataset(all_samples[:1])
_phase, _lbl = _ds_test[0]
print(f"Phase shape : {_phase.shape}")      # should be (1, 224, 224)
print(f"Phase range : [{_phase.min():.3f}, {_phase.max():.3f}]")
print(f"Label       : {_lbl.item()}")
print("PhaseSpectrumDataset OK ✅")

## Cell 4 — Train / Val / Test Splits & DataLoaders

In [ ]:
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15
BATCH_SIZE  = 64
NUM_WORKERS = 2       # Colab is ~2 CPU cores

n_total = len(all_samples)
n_train = int(n_total * TRAIN_RATIO)
n_val   = int(n_total * VAL_RATIO)
n_test  = n_total - n_train - n_val

# Deterministic split using seeded indices
indices = list(range(n_total))
random.shuffle(indices)

train_idx = indices[:n_train]
val_idx   = indices[n_train : n_train + n_val]
test_idx  = indices[n_train + n_val:]

full_dataset = PhaseSpectrumDataset(all_samples, crop_size=CROP_SIZE)

train_ds = Subset(full_dataset, train_idx)
val_ds   = Subset(full_dataset, val_idx)
test_ds  = Subset(full_dataset, test_idx)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

print(f"Train : {len(train_ds)}")
print(f"Val   : {len(val_ds)}")
print(f"Test  : {len(test_ds)}")
print("DataLoaders ready ✅")

## Cell 5 — Kolmogorov–Arnold Network (KAN) Architecture

A KAN replaces fixed activation functions with **learnable** univariate functions on every edge of the network. We parameterize each edge function as a B-spline with trainable coefficients, following the approach of Liu et al., 2024.

For computational tractability on a flattened 224×224 phase spectrum, we first apply a lightweight convolutional feature extractor to reduce dimensionality before feeding into KAN layers.

In [ ]:
class BSplineActivation(nn.Module):
    """
    Learnable univariate activation via B-spline basis functions.
    Each "edge" in the KAN has its own BSplineActivation.
    """

    def __init__(
        self,
        in_features: int,
        num_knots: int = 8,
        spline_order: int = 3,
    ):
        super().__init__()
        self.in_features = in_features
        self.num_knots = num_knots
        self.spline_order = spline_order
        # Total number of basis functions = num_knots + spline_order - 1
        n_bases = num_knots + spline_order - 1
        # Learnable spline coefficients: one set per input feature
        self.coeff = nn.Parameter(
            torch.randn(in_features, n_bases) * 0.1
        )
        # Fixed uniform knot vector on [-1, 1]
        grid = torch.linspace(-1.0, 1.0, num_knots + 2 * spline_order)
        self.register_buffer("grid", grid)

    def _b_spline_basis(self, x: torch.Tensor) -> torch.Tensor:
        """
        Evaluate B-spline basis functions at points x.
        x: (batch, in_features)
        returns: (batch, in_features, n_bases)
        """
        x = x.unsqueeze(-1)              # (B, D, 1)
        grid = self.grid                  # (K,)
        n_bases = self.num_knots + self.spline_order - 1

        # Order-0 basis (piecewise constant)
        bases = ((x >= grid[:-1]) & (x < grid[1:])).float()  # (B, D, K-1)
        # Trim to n_bases
        bases = bases[..., :n_bases + self.spline_order]

        # Recursion for higher orders
        for k in range(1, self.spline_order + 1):
            n = bases.shape[-1] - 1
            left_num  = x - grid[:n].unsqueeze(0).unsqueeze(0)
            left_den  = (grid[k : k + n] - grid[:n]).unsqueeze(0).unsqueeze(0)
            right_num = grid[k + 1 : k + 1 + n].unsqueeze(0).unsqueeze(0) - x
            right_den = (grid[k + 1 : k + 1 + n] - grid[1 : 1 + n]).unsqueeze(0).unsqueeze(0)

            left  = left_num  / (left_den  + 1e-8) * bases[..., :n]
            right = right_num / (right_den + 1e-8) * bases[..., 1 : n + 1]
            bases = left + right

        return bases[..., :self.coeff.shape[1]]   # (B, D, n_bases)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (batch, in_features)
        returns: (batch, in_features)
        """
        # Clamp input to knot range for numerical stability
        x_clamped = torch.clamp(x, -1.0, 1.0)
        basis = self._b_spline_basis(x_clamped)   # (B, D, n_bases)
        # Weighted sum over basis functions per feature
        out = (basis * self.coeff.unsqueeze(0)).sum(dim=-1)  # (B, D)
        return out


class KANLinear(nn.Module):
    """
    A single KAN layer: learnable activation on each edge, then sum.
    Replaces a standard nn.Linear + activation.
    """

    def __init__(self, in_features: int, out_features: int, num_knots: int = 8):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        # One spline activation per (input, output) pair is expensive;
        # instead we use spline on input features, then a linear map.
        self.spline = BSplineActivation(in_features, num_knots=num_knots)
        self.linear = nn.Linear(in_features, out_features)
        # Residual shortcut (SiLU on raw input, like KAN paper)
        self.base_linear = nn.Linear(in_features, out_features)
        self.base_act = nn.SiLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Spline path
        spline_out = self.linear(self.spline(x))
        # Base path (residual)
        base_out = self.base_linear(self.base_act(x))
        return spline_out + base_out


class PhaseKAN(nn.Module):
    """
    Full model: lightweight conv feature extractor → KAN classifier.

    Architecture:
      1) Conv stem (3 blocks): reduces 224×224 → 28×28 → 128 channels
      2) Global Average Pool → 128-dim vector
      3) KAN layers: 128 → 64 → 32 → 1 (logit)
    """

    def __init__(
        self,
        in_channels: int = 1,
        kan_hidden: List[int] = [64, 32],
        num_knots: int = 8,
        dropout: float = 0.3,
    ):
        super().__init__()

        # ── convolutional feature extractor ──────────────────────────────
        self.features = nn.Sequential(
            # Block 1: 1 → 32, 224→112
            nn.Conv2d(in_channels, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.GELU(),
            # Block 2: 32 → 64, 112→56
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.GELU(),
            # Block 3: 64 → 128, 56→28
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.GELU(),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)   # → (B, 128, 1, 1)
        conv_out_dim = 128

        # ── KAN classifier head ──────────────────────────────────────────
        dims = [conv_out_dim] + kan_hidden + [1]
        layers = []
        for i in range(len(dims) - 1):
            layers.append(KANLinear(dims[i], dims[i + 1], num_knots=num_knots))
            if i < len(dims) - 2:
                layers.append(nn.LayerNorm(dims[i + 1]))
                layers.append(nn.Dropout(dropout))
        self.kan_head = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (B, 1, 224, 224) — phase spectrum tensor
        returns: (B, 1) — raw logit (apply sigmoid externally)
        """
        h = self.features(x)       # (B, 128, 28, 28)
        h = self.pool(h)           # (B, 128, 1, 1)
        h = h.flatten(1)           # (B, 128)
        logit = self.kan_head(h)   # (B, 1)
        return logit


# ── instantiate & print summary ─────────────────────────────────────────────
model = PhaseKAN().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {n_params:,}")
print(model)
print("PhaseKAN model ready ✅")

## Cell 6 — Training & Validation Loop

In [ ]:
NUM_EPOCHS    = 20
LEARNING_RATE = 3e-4
WEIGHT_DECAY  = 1e-4
GRAD_CLIP     = 1.0
PATIENCE      = 5               # early stopping patience (val loss)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-6
)

# ── training utilities ───────────────────────────────────────────────────────

def train_one_epoch(model, loader, criterion, optimizer, device, grad_clip=1.0):
    model.train()
    running_loss = 0.0
    n_correct = 0
    n_total   = 0

    for phase_batch, labels in tqdm(loader, desc="  train", leave=False):
        phase_batch = phase_batch.to(device, non_blocking=True)
        labels      = labels.to(device, non_blocking=True).unsqueeze(1)

        optimizer.zero_grad()
        logits = model(phase_batch)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        running_loss += loss.item() * phase_batch.size(0)
        preds = (torch.sigmoid(logits) >= 0.5).float()
        n_correct += (preds == labels).sum().item()
        n_total   += labels.size(0)

    epoch_loss = running_loss / n_total
    epoch_acc  = n_correct / n_total
    return epoch_loss, epoch_acc


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_probs  = []
    all_labels = []

    for phase_batch, labels in tqdm(loader, desc="  eval ", leave=False):
        phase_batch = phase_batch.to(device, non_blocking=True)
        labels      = labels.to(device, non_blocking=True).unsqueeze(1)

        logits = model(phase_batch)
        loss   = criterion(logits, labels)

        running_loss += loss.item() * phase_batch.size(0)
        probs = torch.sigmoid(logits).cpu()
        all_probs.append(probs)
        all_labels.append(labels.cpu())

    all_probs  = torch.cat(all_probs).numpy().flatten()
    all_labels = torch.cat(all_labels).numpy().flatten()
    n_total    = len(all_labels)
    epoch_loss = running_loss / n_total
    preds      = (all_probs >= 0.5).astype(float)
    epoch_acc  = accuracy_score(all_labels, preds)
    return epoch_loss, epoch_acc, all_probs, all_labels


# ── training loop ────────────────────────────────────────────────────────────
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_loss = float("inf")
patience_counter = 0
best_model_state = None

print(f"\n{'='*60}")
print(f" Training — {NUM_EPOCHS} epochs, lr={LEARNING_RATE}, wd={WEIGHT_DECAY}")
print(f"{'='*60}\n")

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, DEVICE, GRAD_CLIP
    )
    val_loss, val_acc, _, _ = evaluate(
        model, val_loader, criterion, DEVICE
    )
    scheduler.step()

    dt = time.time() - t0
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    star = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        star = " ★"
    else:
        patience_counter += 1

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} │ "
        f"train loss {train_loss:.4f} acc {train_acc:.4f} │ "
        f"val loss {val_loss:.4f} acc {val_acc:.4f} │ "
        f"{dt:.1f}s{star}"
    )

    if patience_counter >= PATIENCE:
        print(f"\n⏹  Early stopping at epoch {epoch} (patience={PATIENCE})")
        break

# Restore best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    model.to(DEVICE)
print("\nTraining complete ✅  (best model restored)")

## Cell 7 — Test-Set Evaluation & IEEE-Standard Metrics

In [ ]:
test_loss, test_acc, test_probs, test_labels = evaluate(
    model, test_loader, criterion, DEVICE
)
test_preds = (test_probs >= 0.5).astype(float)

# ── compute all metrics ──────────────────────────────────────────────────────
precision = precision_score(test_labels, test_preds, zero_division=0)
recall    = recall_score(test_labels, test_preds, zero_division=0)
f1        = f1_score(test_labels, test_preds, zero_division=0)
roc_auc   = roc_auc_score(test_labels, test_probs)

print(f"\n{'='*60}")
print(f" Test-Set Results")
print(f"{'='*60}")
print(f"  Loss      : {test_loss:.4f}")
print(f"  Accuracy  : {test_acc:.4f}  ({test_acc*100:.2f}%)")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print(f"  ROC-AUC   : {roc_auc:.4f}")
print(f"{'='*60}\n")

print("Classification Report:")
print(classification_report(
    test_labels, test_preds,
    target_names=["Real", "Fake"], digits=4
))

## Cell 8 — Visualisation: ROC Curve, Confusion Matrix, Learning Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── (a) ROC Curve ────────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(test_labels, test_probs)
axes[0].plot(fpr, tpr, color="royalblue", lw=2,
             label=f"KAN (AUC = {roc_auc:.4f})")
axes[0].plot([0, 1], [0, 1], "--", color="grey", lw=1)
axes[0].set_xlabel("False Positive Rate", fontsize=12)
axes[0].set_ylabel("True Positive Rate", fontsize=12)
axes[0].set_title("ROC Curve", fontsize=14, fontweight="bold")
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)

# ── (b) Confusion Matrix ────────────────────────────────────────────────────
cm = confusion_matrix(test_labels, test_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=["Real", "Fake"])
disp.plot(ax=axes[1], cmap="Blues", colorbar=False)
axes[1].set_title("Confusion Matrix", fontsize=14, fontweight="bold")

# ── (c) Learning Curves ─────────────────────────────────────────────────────
epochs_range = list(range(1, len(history["train_loss"]) + 1))
axes[2].plot(epochs_range, history["train_loss"], label="Train Loss", lw=2)
axes[2].plot(epochs_range, history["val_loss"], label="Val Loss", lw=2)
ax2 = axes[2].twinx()
ax2.plot(epochs_range, history["train_acc"], "--", label="Train Acc", lw=1.5, color="green")
ax2.plot(epochs_range, history["val_acc"], "--", label="Val Acc", lw=1.5, color="red")
axes[2].set_xlabel("Epoch", fontsize=12)
axes[2].set_ylabel("Loss", fontsize=12)
ax2.set_ylabel("Accuracy", fontsize=12)
axes[2].set_title("Learning Curves", fontsize=14, fontweight="bold")
lines1, labels1 = axes[2].get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
axes[2].legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc="center right")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("kan_deepfake_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Results figure saved → kan_deepfake_results.png ✅")

## Cell 9 — Save Model & Training Summary

In [ ]:
SAVE_DIR = Path("runs/kan_deepfake")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Save model checkpoint
ckpt_path = SAVE_DIR / "best_phase_kan.pt"
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "epoch": len(history["train_loss"]),
    "best_val_loss": best_val_loss,
    "config": {
        "crop_size": CROP_SIZE,
        "batch_size": BATCH_SIZE,
        "lr": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "grad_clip": GRAD_CLIP,
        "seed": SEED,
    },
}, ckpt_path)
print(f"Checkpoint saved → {ckpt_path}")

# Save training summary JSON (for paper appendix)
summary = {
    "dataset": "DeepDetect-2025",
    "model": "PhaseKAN (Conv stem + KAN classifier)",
    "parameters": n_params,
    "crop_size": CROP_SIZE,
    "train_samples": len(train_ds),
    "val_samples": len(val_ds),
    "test_samples": len(test_ds),
    "epochs_trained": len(history["train_loss"]),
    "best_val_loss": best_val_loss,
    "test_metrics": {
        "loss": round(test_loss, 4),
        "accuracy": round(test_acc, 4),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1_score": round(f1, 4),
        "roc_auc": round(roc_auc, 4),
    },
    "history": history,
}
summary_path = SAVE_DIR / "training_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"Summary saved  → {summary_path}")
print("\n🏁 Pipeline complete.")